In [1]:
import dataclasses

## 1. dataclass 예제
# ----------------------------------------------------

@dataclasses.dataclass # decorator
class DataUser:
    """타입 힌트를 사용하지만, 런타임 유효성 검사는 없음."""
    id: int
    name: str
    is_active: bool = True

print("="*15 + " dataclass 결과 " + "="*15)

=============== dataclass 결과 ===============


In [2]:
# (1) 올바른 데이터
user_data = DataUser(id=1, name="Alice")
print(f"✅ 유효한 데이터 (정상): {user_data}")


✅ 유효한 데이터 (정상): DataUser(id=1, name='Alice', is_active=True)


In [3]:
# (2) 유효하지 않은 데이터 (str 대신 int를 할당)
# dataclass는 오류를 발생시키지 않고 값을 그대로 저장함 (타입 검사 X)
invalid_user = DataUser(id=2, name=123)
print(f"⚠️ 유효하지 않은 데이터 (저장됨): {invalid_user}")

⚠️ 유효하지 않은 데이터 (저장됨): DataUser(id=2, name=123, is_active=True)


## 클래스 타입과 Pydantic (FastAPI의 핵심)

FastAPI에서는 사용자가 정의한 클래스 자체를 타입으로 사용합니다.

특히 데이터 검증 라이브러리인 Pydantic 모델을 타입 힌트로 지정하면, FastAPI가 자동으로 JSON 데이터를 파싱하고 검증해줍니다.

In [12]:
from pydantic import BaseModel

class Item(BaseModel):
    name: str
    price: float
    is_offer: bool | None = None


In [15]:
item=Item(name="Foo", price=50.2)
item.name

'Foo'

In [10]:
# FastAPI에서 이런 식으로 사용하게 됩니다.
def create_item(item: Item):
    return {"item_name": item.name, "item_price": item.price}

In [11]:
create_item(item=Item(name="Foo", price=50.2))

{'item_name': 'Foo', 'item_price': 50.2}

In [16]:
from pydantic import ValidationError

## 2. Pydantic BaseModel 예제
# ----------------------------------------------------

class PydanticUser(BaseModel):
    """타입 힌트를 사용하며, 런타임에 유효성 검사 및 강제 변환 수행."""
    id: int
    name: str
    is_active: bool = True

print("\n" + "="*15 + " Pydantic 결과 " + "="*15)

# (1) 올바른 데이터
p_user_data = PydanticUser(id=1, name="Bob")
print(f"✅ 유효한 데이터 (정상): {p_user_data}")

# (2) 타입 강제 변환 (str '2'가 int로 자동 변환됨)
p_user_coerce = PydanticUser(id='2', name="Charlie")
print(f"🔄 타입 강제 변환 (정상): {p_user_coerce}")

# (3) 유효성 검사 실패 (유효한 bool 값이 아님)
try:
    PydanticUser(id=3, name="David", is_active="not_a_bool")
except ValidationError as e:
    print(f"❌ 유효성 검사 실패: \n{e.errors()[0]}")


=============== Pydantic 결과 ===============
✅ 유효한 데이터 (정상): id=1 name='Bob' is_active=True
🔄 타입 강제 변환 (정상): id=2 name='Charlie' is_active=True
❌ 유효성 검사 실패: 
{'type': 'bool_parsing', 'loc': ('is_active',), 'msg': 'Input should be a valid boolean, unable to interpret input', 'input': 'not_a_bool', 'url': 'https://errors.pydantic.dev/2.12/v/bool_parsing'}
